In [1]:
import random
from collections import Counter

# ---- Card representation ----
RANKS = list(range(2, 15))
SUITS = ['s', 'h', 'd', 'c']
DECK = [(r, s) for r in RANKS for s in SUITS]

RANK_NAMES = {2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8',
              9:'9', 10:'10', 11:'J', 12:'Q', 13:'K', 14:'A'}

def card_str(card):
    r, s = card
    return f"{RANK_NAMES[r]}{s}"

def hand_str(hand):
    return '[' + ', '.join(card_str(c) for c in hand) + ']'

# ---- Hand evaluation (O(n), no combinations) ----

def best_hand_rank(cards):
    if not cards:
        return (0, [])

    ranks = sorted([r for r, s in cards], reverse=True)
    rank_counts = Counter(r for r, s in cards)
    suit_counts = Counter(s for r, s in cards)
    rank_set = set(ranks)

    flush_suit = next((s for s, c in suit_counts.items() if c >= 5), None)
    flush_ranks = sorted((r for r, s in cards if s == flush_suit), reverse=True) if flush_suit else []

    def best_straight(rs):
        rs = rs | ({1} if 14 in rs else set())
        for top in range(14, 4, -1):
            if set(range(top - 4, top + 1)) <= rs:
                return top
        return None

    straight_top = best_straight(rank_set)
    sf_top = best_straight(set(flush_ranks)) if flush_suit else None

    counts_desc = sorted(rank_counts.items(), key=lambda x: (x[1], x[0]), reverse=True)
    by_count = lambda n: [r for r, c in counts_desc if c == n]

    quads = by_count(4)
    trips = by_count(3)
    pairs = by_count(2)

    def kickers(exclude, n):
        return sorted([r for r in ranks if r not in exclude], reverse=True)[:n]

    if sf_top:
        return (8, [sf_top])
    if quads:
        q = quads[0]; return (7, [q] + kickers({q}, 1))
    if trips:
        t = trips[0]
        pair_rank = trips[1] if len(trips) > 1 else (pairs[0] if pairs else None)
        if pair_rank is not None:
            return (6, [t, pair_rank])
    if flush_suit:
        return (5, flush_ranks[:5])
    if straight_top:
        return (4, [straight_top])
    if trips:
        t = trips[0]; return (3, [t] + kickers({t}, 2))
    if len(pairs) >= 2:
        p1, p2 = pairs[0], pairs[1]; return (2, [p1, p2] + kickers({p1, p2}, 1))
    if pairs:
        p = pairs[0]; return (1, [p] + kickers({p}, 3))
    return (0, ranks[:5])

# ---- Flush strategy ----
# Pick 1: take any card (first card in deck)
# Picks 2-5: take first card matching the suit of pick 1

def play_flush_strategy(deck):
    """
    Play one game with flush strategy on a fixed deck.
    Returns (my_hand, dealer_hand, win)
    """
    my_hand = []
    dealer_hand = []
    remaining = list(deck)

    # Pick 1: take whatever is first
    first_card = remaining.pop(0)
    my_hand.append(first_card)
    flush_suit = first_card[1]

    # Picks 2-5: chase the flush suit
    for _ in range(4):
        for i, card in enumerate(remaining):
            if card[1] == flush_suit:
                dealer_hand += remaining[:i]
                my_hand.append(card)
                remaining = remaining[i+1:]
                break

    # Top up dealer to 8 cards
    needed = max(0, 8 - len(dealer_hand))
    final_dealer = dealer_hand + remaining[:needed]

    my_rank = best_hand_rank(my_hand)
    dealer_rank = best_hand_rank(final_dealer)
    win = my_rank > dealer_rank

    return my_hand, final_dealer, win

# ---- Run simulation ----

N_SIMULATIONS = 5000

wins = 0
hand_type_counts = Counter()
final_hands = []

for _ in range(N_SIMULATIONS):
    deck = list(DECK)
    random.shuffle(deck)
    my_hand, dealer_hand, win = play_flush_strategy(deck)
    if win:
        wins += 1
    hand_type_counts[best_hand_rank(my_hand)[0]] += 1
    final_hands.append(my_hand)

HAND_NAMES = {
    0: 'High Card', 1: 'One Pair', 2: 'Two Pair', 3: 'Three of a Kind',
    4: 'Straight',  5: 'Flush',   6: 'Full House', 7: 'Four of a Kind',
    8: 'Straight Flush'
}

print("=" * 50)
print("FLUSH STRATEGY: take any first card, chase suit")
print("=" * 50)
print(f"\nSimulations:  {N_SIMULATIONS}")
print(f"Win rate:     {wins/N_SIMULATIONS:.3f}  ({wins}/{N_SIMULATIONS})")

print("\nPlayer hand distribution:")
for cat in range(8, -1, -1):
    count = hand_type_counts[cat]
    if count > 0:
        pct = count / N_SIMULATIONS * 100
        print(f"  {HAND_NAMES[cat]:<20} {count:>5}  ({pct:.1f}%)")

# Average hand — show most common final hand per category
print("\nMost common outcome hands (sample of 5 per category):")
from collections import defaultdict
by_category = defaultdict(list)
for hand in final_hands:
    cat = best_hand_rank(hand)[0]
    by_category[cat].append(hand)

for cat in range(8, -1, -1):
    hands = by_category[cat]
    if hands:
        sample = hands[:5]
        print(f"\n  {HAND_NAMES[cat]}:")
        for h in sample:
            print(f"    {hand_str(h)}")

FLUSH STRATEGY: take any first card, chase suit

Simulations:  5000
Win rate:     0.452  (2262/5000)

Player hand distribution:
  Straight Flush          45  (0.9%)
  Flush                 4955  (99.1%)

Most common outcome hands (sample of 5 per category):

  Straight Flush:
    [Js, 8s, 9s, 7s, 10s]
    [Jh, Kh, 10h, Qh, Ah]
    [2s, 5s, As, 3s, 4s]
    [7h, 4h, 5h, 6h, 3h]
    [Ac, 10c, Qc, Kc, Jc]

  Flush:
    [4c, Jc, Qc, 5c, 6c]
    [9s, Js, 5s, As, 4s]
    [4h, Jh, 9h, 8h, 7h]
    [8h, Kh, Ah, 6h, 3h]
    [As, 2s, 8s, 4s, 3s]
